# 10-Fold Transformer-Based Heterogeneous GNN Training for CXR KG

This notebook extends `hetero_graphsage_10fold_training.ipynb` by testing transformer-style GNN layers on the same fold-specific chest X-ray knowledge graphs.

Ablations:

- `HGTConv`: heterogeneous graph transformer.
- `HANConv`: heterogeneous attention network.
- `TransformerConv`: edge-type-specific transformer convolution wrapped by `HeteroConv`.
- Query retrieval `top_k`: `50`, `100`, `200`, `500`.

Outputs are saved under `/data/liangz2/openi/multi_kg/kg_transformer_gnn_10fold/{model_name}/top_k_{k}/fold{fold}/` so each model/top-k/fold run is isolated and resumable.

In [1]:
# Optional dependency installation. Set True only in a fresh cloud environment.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys
    packages = [
        "torch",
        "torch_geometric",
        "faiss-cpu",
        "scikit-learn",
        "pandas",
        "numpy",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

In [2]:
import copy
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import HeteroConv, HGTConv, HANConv, TransformerConv
except ImportError as exc:
    raise ImportError(
        "Install torch_geometric before running this notebook. "
        "In your diffuser2 environment, try: pip install torch_geometric faiss-cpu"
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = Path("/data/liangz2/openi/multi_kg") if Path("/data/liangz2/openi/multi_kg").exists() else Path.cwd()
KG_10FOLD_DIR = DATA_ROOT / "kg_10fold"
OUTPUT_ROOT = DATA_ROOT / "kg_transformer_gnn_10fold"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Main ablation settings.
MODEL_NAMES = ["hgtconv", "hanconv", "transformerconv"]
QUERY_TOP_K_VALUES = [50, 100, 200, 500]
FOLD_INDICES = list(range(10))

SKIP_COMPLETED = True
FORCE_RERUN = []  # Optional list of (model_name, top_k, fold), e.g. [("hgtconv", 100, 0)]

# Training config. These are the defaults for HGT/HAN.
VAL_FRAC = 0.10
HIDDEN_DIM = 256
NUM_LAYERS = 2
NUM_HEADS = 4
DROPOUT = 0.20
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100
PATIENCE = 12
GRAD_CLIP_NORM = 5.0

# Per-model overrides. TransformerConv is much more memory intensive because it
# computes attention per edge type, especially on image-image similar_to edges.
# Keep the transformerconv setting conservative first; increase later only if it fits.
MODEL_CONFIGS = {
    "hgtconv": {
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "dropout": DROPOUT,
        "max_train_similar_edges_per_node": None,
        "query_inference_chunk_size": QUERY_INFERENCE_CHUNK_SIZE if "QUERY_INFERENCE_CHUNK_SIZE" in globals() else 512,
    },
    "hanconv": {
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "dropout": DROPOUT,
        "max_train_similar_edges_per_node": None,
        "query_inference_chunk_size": QUERY_INFERENCE_CHUNK_SIZE if "QUERY_INFERENCE_CHUNK_SIZE" in globals() else 512,
    },
    "transformerconv": {
        "hidden_dim": 128,
        "num_layers": 1,
        "num_heads": 2,
        "dropout": DROPOUT,
        "max_train_similar_edges_per_node": 50,
        "query_inference_chunk_size": 128,
    },
}


def get_model_config(model_name: str) -> Dict:
    base = {
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "dropout": DROPOUT,
        "max_train_similar_edges_per_node": None,
        "query_inference_chunk_size": QUERY_INFERENCE_CHUNK_SIZE if "QUERY_INFERENCE_CHUNK_SIZE" in globals() else 512,
    }
    base.update(MODEL_CONFIGS.get(model_name.lower(), {}))
    return base

# Decoding config.
DECODER = "dot"  # "dot" or "mlp"

# Query inference config.
TEST_QUERY_BATCH_SIZE = 512
QUERY_INFERENCE_CHUNK_SIZE = 512

# Leakage control. Keep False for the primary experiment.
INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING = False

SCORING_LABELS = (
    "normal",
    "enlarged cardiomediastinum", "cardiomegaly", "lung opacity", "lung lesion", "edema",
    "consolidation", "pneumonia", "atelectasis", "pneumothorax",
    "pleural effusion", "pleural other", "fracture", "support devices",
)
TARGET_COLUMNS = ["y_" + label.replace(" ", "_") for label in SCORING_LABELS]
FINDING_LABELS = SCORING_LABELS[1:]

print("DEVICE", DEVICE)
print("DATA_ROOT", DATA_ROOT)
print("KG_10FOLD_DIR", KG_10FOLD_DIR)
print("OUTPUT_ROOT", OUTPUT_ROOT)
print("MODEL_NAMES", MODEL_NAMES)
print("QUERY_TOP_K_VALUES", QUERY_TOP_K_VALUES)
print("INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING", INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING)

DEVICE cuda
DATA_ROOT /data/liangz2/openi/multi_kg
KG_10FOLD_DIR /data/liangz2/openi/multi_kg/kg_10fold
OUTPUT_ROOT /data/liangz2/openi/multi_kg/kg_transformer_gnn_10fold
MODEL_NAMES ['hgtconv', 'hanconv', 'transformerconv']
QUERY_TOP_K_VALUES [50, 100, 200, 500]
INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING False


## Metrics and Utility Functions

In [3]:
def sigmoid_np(logits: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-logits))


def evaluate_prob_matrix(probs: np.ndarray, targets: np.ndarray, thresholds: Optional[np.ndarray] = None) -> pd.DataFrame:
    y = targets.astype(int)
    if thresholds is None:
        thresholds = np.full((probs.shape[1],), 0.5, dtype=np.float32)
    rows = []
    for j, label in enumerate(SCORING_LABELS):
        if len(np.unique(y[:, j])) < 2:
            continue
        pred = (probs[:, j] >= thresholds[j]).astype(int)
        rows.append({
            "label": label,
            "auroc": roc_auc_score(y[:, j], probs[:, j]),
            "auprc": average_precision_score(y[:, j], probs[:, j]),
            "f1": f1_score(y[:, j], pred, zero_division=0),
            "threshold": float(thresholds[j]),
            "support": int(y[:, j].sum()),
        })
    return pd.DataFrame(rows)


def macro_metric(metrics: pd.DataFrame, metric: str = "auroc", disease_only: bool = True) -> float:
    if metrics.empty:
        return float("nan")
    frame = metrics[metrics["label"] != "normal"] if disease_only else metrics
    if frame.empty:
        return float("nan")
    return float(frame[metric].mean())


def tune_f1_thresholds(probs: np.ndarray, targets: np.ndarray, grid: Optional[np.ndarray] = None) -> np.ndarray:
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    thresholds = np.full((probs.shape[1],), 0.5, dtype=np.float32)
    y = targets.astype(int)
    for j in range(probs.shape[1]):
        if len(np.unique(y[:, j])) < 2:
            continue
        best_f1, best_thr = -1.0, 0.5
        for thr in grid:
            pred = (probs[:, j] >= thr).astype(int)
            value = f1_score(y[:, j], pred, zero_division=0)
            if value > best_f1:
                best_f1, best_thr = value, float(thr)
        thresholds[j] = best_thr
    return thresholds


def metric_summary(metrics: pd.DataFrame, fold: int, method: str, extra: Optional[Dict] = None) -> Dict:
    disease = metrics[metrics["label"] != "normal"] if not metrics.empty else metrics
    out = {
        "fold": int(fold),
        "method": method,
        "disease_macro_auroc": float(disease["auroc"].mean()) if len(disease) else float("nan"),
        "disease_macro_auprc": float(disease["auprc"].mean()) if len(disease) else float("nan"),
        "disease_macro_f1": float(disease["f1"].mean()) if len(disease) else float("nan"),
        "all_macro_auroc": float(metrics["auroc"].mean()) if len(metrics) else float("nan"),
        "all_macro_auprc": float(metrics["auprc"].mean()) if len(metrics) else float("nan"),
        "all_macro_f1": float(metrics["f1"].mean()) if len(metrics) else float("nan"),
        "num_scored_labels": int(len(metrics)),
    }
    if extra:
        out.update(extra)
    return out


def save_json(obj: Dict, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")

## Load Fold Graphs

The fold graph is expected at `kg_10fold/fold{i}/kg_heterodata_fold{i}.pt`. The graph should contain training image nodes and their labels. Test images are added later as query nodes connected only through `similar_to` retrieval edges.

In [4]:
def torch_load_any(path: Path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def dict_to_heterodata(obj: Dict) -> HeteroData:
    data = HeteroData()
    for node_type, x in obj.get("x_dict", {}).items():
        data[node_type].x = x.float()
        node_ids = obj.get("node_ids", {}).get(node_type)
        if node_ids is not None:
            data[node_type].node_id = list(node_ids)
        if node_type in obj.get("y_dict", {}):
            data[node_type].y = obj["y_dict"][node_type].float()
    for etype, edge_index in obj.get("edge_index_dict", {}).items():
        data[etype].edge_index = edge_index.long()
        edge_weight = obj.get("edge_weight_dict", {}).get(etype)
        if edge_weight is not None:
            data[etype].edge_weight = edge_weight.float()
    return data


def load_fold_data(fold: int) -> Tuple[HeteroData, pd.DataFrame, pd.DataFrame, np.ndarray]:
    fold_dir = KG_10FOLD_DIR / f"fold{fold}"
    kg_path = fold_dir / f"kg_heterodata_fold{fold}.pt"
    train_meta_path = fold_dir / f"train_metadata_fold{fold}.csv"
    test_meta_path = fold_dir / f"test_query_metadata_fold{fold}.csv"
    test_emb_path = fold_dir / f"test_query_embeddings_fold{fold}.npy"
    if not kg_path.exists():
        raise FileNotFoundError(f"Missing fold KG file: {kg_path}")
    obj = torch_load_any(kg_path, map_location="cpu")
    data = obj if isinstance(obj, HeteroData) else dict_to_heterodata(obj)
    train_df = pd.read_csv(train_meta_path)
    test_df = pd.read_csv(test_meta_path)
    test_embeddings = np.load(test_emb_path).astype("float32")
    return data, train_df, test_df, test_embeddings


def label_names_from_data(data: HeteroData) -> List[str]:
    if hasattr(data["label"], "label_name"):
        return [str(x) for x in data["label"].label_name]
    if hasattr(data["label"], "node_id"):
        names = []
        for node_id in data["label"].node_id:
            text = str(node_id)
            names.append(text.split("label::", 1)[-1] if text.startswith("label::") else text)
        return names
    raise RuntimeError("Label node names are missing from HeteroData. Regenerate KG with label_name/node_id metadata.")


def label_indices_for_scoring(data: HeteroData) -> List[int]:
    label_names = label_names_from_data(data)
    lookup = {name: i for i, name in enumerate(label_names)}
    missing = [label for label in SCORING_LABELS if label not in lookup]
    if missing:
        raise RuntimeError(f"Missing label nodes: {missing}. Available labels: {label_names}")
    return [lookup[label] for label in SCORING_LABELS]


def split_train_val_indices(num_images: int, val_frac: float, seed: int) -> Tuple[torch.Tensor, torch.Tensor]:
    rng = np.random.default_rng(seed)
    indices = np.arange(num_images)
    rng.shuffle(indices)
    val_size = max(1, int(round(num_images * val_frac)))
    val_idx = np.sort(indices[:val_size])
    train_idx = np.sort(indices[val_size:])
    return torch.from_numpy(train_idx).long(), torch.from_numpy(val_idx).long()

## Transformer-Based Heterogeneous GNN Models

All three models share the same input projections and image-label decoder. The difference is the message-passing layer:

- `HGTConv`: native heterogeneous graph transformer.
- `HANConv`: semantic attention over heterogeneous neighborhoods.
- `TransformerConv`: per-edge-type transformer convolution inside `HeteroConv`.

The `has_finding` and `rev_has_finding` edges are excluded from message passing by default to avoid label leakage. They remain supervision targets only.

In [5]:
def merge_layer_output(prev_z: Dict[str, torch.Tensor], out_z: Dict[str, torch.Tensor], norm_dict: nn.ModuleDict, dropout: float, training: bool) -> Dict[str, torch.Tensor]:
    next_z = {}
    for node_type, x in prev_z.items():
        y = out_z.get(node_type)
        if y is None:
            next_z[node_type] = x
            continue
        y = F.relu(y)
        y = norm_dict[node_type](y)
        y = F.dropout(y, p=dropout, training=training)
        next_z[node_type] = x + y if x.shape == y.shape else y
    return next_z


class BaseImageLabelDecoder(nn.Module):
    def _init_decoder(self, hidden_dim: int, dropout: float, decoder: str):
        self.decoder = decoder
        if decoder == "mlp":
            self.decoder_mlp = nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1),
            )
        elif decoder != "dot":
            raise ValueError(f"Unknown decoder: {decoder}")

    def decode_image_label_logits(self, z_dict, label_indices: Sequence[int]) -> torch.Tensor:
        image_z = z_dict["image"]
        label_z = z_dict["label"][torch.as_tensor(label_indices, device=image_z.device)]
        if self.decoder == "dot":
            return image_z @ label_z.t()
        image_expanded = image_z[:, None, :].expand(-1, len(label_indices), -1)
        label_expanded = label_z[None, :, :].expand(image_z.size(0), -1, -1)
        pair = torch.cat([image_expanded, label_expanded], dim=-1)
        return self.decoder_mlp(pair).squeeze(-1)


def make_hgt_conv(hidden_dim: int, metadata, heads: int):
    try:
        return HGTConv(hidden_dim, hidden_dim, metadata, heads=heads, group="sum")
    except TypeError as exc:
        # Older torch_geometric versions do not expose the HGTConv group argument.
        if "group" in str(exc):
            return HGTConv(hidden_dim, hidden_dim, metadata, heads=heads)
        raise


class HeteroHGT(BaseImageLabelDecoder):
    def __init__(self, metadata, input_dims: Dict[str, int], hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, heads=NUM_HEADS, dropout=DROPOUT, decoder=DECODER):
        super().__init__()
        node_types, edge_types = metadata
        self.dropout = dropout
        self.proj = nn.ModuleDict({node_type: nn.Linear(int(input_dims[node_type]), hidden_dim) for node_type in node_types if node_type in input_dims})
        self.convs = nn.ModuleList([make_hgt_conv(hidden_dim, metadata, heads) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.ModuleDict({node_type: nn.LayerNorm(hidden_dim) for node_type in self.proj}) for _ in range(num_layers)])
        self._init_decoder(hidden_dim, dropout, decoder)

    def forward(self, x_dict, edge_index_dict):
        z = {node_type: F.relu(self.proj[node_type](x.float())) for node_type, x in x_dict.items() if node_type in self.proj}
        for conv, norm_dict in zip(self.convs, self.norms):
            out = conv(z, edge_index_dict)
            z = merge_layer_output(z, out, norm_dict, self.dropout, self.training)
        return z


class HeteroHAN(BaseImageLabelDecoder):
    def __init__(self, metadata, input_dims: Dict[str, int], hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, heads=NUM_HEADS, dropout=DROPOUT, decoder=DECODER):
        super().__init__()
        node_types, edge_types = metadata
        self.dropout = dropout
        self.proj = nn.ModuleDict({node_type: nn.Linear(int(input_dims[node_type]), hidden_dim) for node_type in node_types if node_type in input_dims})
        in_channels = {node_type: hidden_dim for node_type in node_types if node_type in input_dims}
        self.convs = nn.ModuleList([HANConv(in_channels, hidden_dim, metadata=metadata, heads=heads, dropout=dropout) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.ModuleDict({node_type: nn.LayerNorm(hidden_dim) for node_type in self.proj}) for _ in range(num_layers)])
        self._init_decoder(hidden_dim, dropout, decoder)

    def forward(self, x_dict, edge_index_dict):
        z = {node_type: F.relu(self.proj[node_type](x.float())) for node_type, x in x_dict.items() if node_type in self.proj}
        for conv, norm_dict in zip(self.convs, self.norms):
            out = conv(z, edge_index_dict)
            z = merge_layer_output(z, out, norm_dict, self.dropout, self.training)
        return z


class HeteroTransformerConv(BaseImageLabelDecoder):
    def __init__(self, metadata, input_dims: Dict[str, int], hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, heads=NUM_HEADS, dropout=DROPOUT, decoder=DECODER):
        super().__init__()
        node_types, edge_types = metadata
        self.dropout = dropout
        self.proj = nn.ModuleDict({node_type: nn.Linear(int(input_dims[node_type]), hidden_dim) for node_type in node_types if node_type in input_dims})
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: TransformerConv((hidden_dim, hidden_dim), hidden_dim, heads=heads, concat=False, dropout=dropout)
                for edge_type in edge_types
            }, aggr="sum")
            self.convs.append(conv)
            self.norms.append(nn.ModuleDict({node_type: nn.LayerNorm(hidden_dim) for node_type in self.proj}))
        self._init_decoder(hidden_dim, dropout, decoder)

    def forward(self, x_dict, edge_index_dict):
        z = {node_type: F.relu(self.proj[node_type](x.float())) for node_type, x in x_dict.items() if node_type in self.proj}
        for conv, norm_dict in zip(self.convs, self.norms):
            out = conv(z, edge_index_dict)
            z = merge_layer_output(z, out, norm_dict, self.dropout, self.training)
        return z


def build_model(model_name: str, metadata, input_dims: Dict[str, int], cfg: Optional[Dict] = None):
    model_name = model_name.lower()
    cfg = cfg or get_model_config(model_name)
    kwargs = {
        "hidden_dim": int(cfg["hidden_dim"]),
        "num_layers": int(cfg["num_layers"]),
        "heads": int(cfg["num_heads"]),
        "dropout": float(cfg["dropout"]),
        "decoder": DECODER,
    }
    if model_name == "hgtconv":
        return HeteroHGT(metadata, input_dims, **kwargs)
    if model_name == "hanconv":
        return HeteroHAN(metadata, input_dims, **kwargs)
    if model_name == "transformerconv":
        return HeteroTransformerConv(metadata, input_dims, **kwargs)
    raise ValueError(f"Unknown model_name={model_name}. Expected one of: hgtconv, hanconv, transformerconv")


def keep_first_edges_per_src(edge_index: torch.Tensor, max_edges_per_src: Optional[int]) -> torch.Tensor:
    if max_edges_per_src is None or max_edges_per_src <= 0 or edge_index.numel() == 0:
        return torch.arange(edge_index.size(1), dtype=torch.long)
    src_nodes = edge_index[0].cpu().tolist()
    counts = {}
    keep = []
    for col, src in enumerate(src_nodes):
        used = counts.get(src, 0)
        if used < max_edges_per_src:
            keep.append(col)
            counts[src] = used + 1
    return torch.tensor(keep, dtype=torch.long)


def cap_similar_edges_in_data(data: HeteroData, max_edges_per_src: Optional[int]) -> HeteroData:
    if max_edges_per_src is None:
        return data
    for edge_type in [("image", "similar_to", "image"), ("image", "rev_similar_to", "image")]:
        if edge_type not in data.edge_types or not hasattr(data[edge_type], "edge_index"):
            continue
        edge_index = data[edge_type].edge_index.cpu()
        keep = keep_first_edges_per_src(edge_index, max_edges_per_src)
        old_edges = edge_index.size(1)
        data[edge_type].edge_index = edge_index[:, keep]
        if hasattr(data[edge_type], "edge_weight"):
            data[edge_type].edge_weight = data[edge_type].edge_weight.cpu()[keep]
        print(f"Capped {edge_type}: {old_edges} -> {data[edge_type].edge_index.size(1)} edges")
    return data


def message_passing_edge_index_dict(data: HeteroData, include_has_finding: bool = INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING):
    edge_dict = {}
    for edge_type in data.edge_types:
        relation = edge_type[1]
        if not include_has_finding and (relation == "has_finding" or relation == "rev_has_finding"):
            continue
        if hasattr(data[edge_type], "edge_index"):
            edge_dict[edge_type] = data[edge_type].edge_index
    return edge_dict


def input_dims_from_data(data: HeteroData) -> Dict[str, int]:
    return {node_type: int(data[node_type].x.size(-1)) for node_type in data.node_types if hasattr(data[node_type], "x")}


def pos_weight_from_targets(y: torch.Tensor) -> torch.Tensor:
    pos = y.sum(dim=0)
    neg = y.size(0) - pos
    return torch.clamp(neg / torch.clamp(pos, min=1.0), min=1.0, max=50.0)

## Query Graph Construction

For each test image, we add a query image node and connect it to the `top_k` nearest training image nodes through `similar_to` edges. The query node has no `has_finding` edges. To keep the transformer models memory-safe at `top_k=500`, test inference is performed in chunks.

In [6]:
def l2_normalize_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)).astype("float32")


def topk_inner_product(train_embeddings: np.ndarray, query_embeddings: np.ndarray, top_k: int, batch_size: int = TEST_QUERY_BATCH_SIZE):
    train = l2_normalize_np(train_embeddings.astype("float32"))
    query = l2_normalize_np(query_embeddings.astype("float32"))
    all_sims, all_idx = [], []
    try:
        import faiss
        index = faiss.IndexFlatIP(train.shape[1])
        index.add(np.ascontiguousarray(train))
        for start in tqdm(range(0, len(query), batch_size), desc=f"query FAISS top_k={top_k}"):
            sims, idx = index.search(np.ascontiguousarray(query[start:start + batch_size]), min(top_k, len(train)))
            all_sims.append(sims.astype("float32"))
            all_idx.append(idx.astype("int64"))
    except ImportError:
        for start in tqdm(range(0, len(query), batch_size), desc=f"query dense top_k={top_k}"):
            scores = query[start:start + batch_size] @ train.T
            kth = min(top_k, scores.shape[1]) - 1
            idx = np.argpartition(-scores, kth=kth, axis=1)[:, :top_k]
            sorted_order = np.take_along_axis(scores, idx, axis=1).argsort(axis=1)[:, ::-1]
            idx = np.take_along_axis(idx, sorted_order, axis=1)
            sims = np.take_along_axis(scores, idx, axis=1)
            all_sims.append(sims.astype("float32"))
            all_idx.append(idx.astype("int64"))
    return np.concatenate(all_sims, axis=0), np.concatenate(all_idx, axis=0)


def append_edge_index(data: HeteroData, edge_type, edge_index_new: torch.Tensor, edge_weight_new: Optional[torch.Tensor] = None):
    store = data[edge_type]
    if hasattr(store, "edge_index"):
        store.edge_index = torch.cat([store.edge_index.cpu(), edge_index_new.cpu()], dim=1)
    else:
        store.edge_index = edge_index_new.cpu()
    if edge_weight_new is not None:
        if hasattr(store, "edge_weight"):
            store.edge_weight = torch.cat([store.edge_weight.cpu(), edge_weight_new.cpu()], dim=0)
        else:
            store.edge_weight = edge_weight_new.cpu()


def target_matrix_from_metadata(frame: pd.DataFrame) -> np.ndarray:
    missing = [col for col in TARGET_COLUMNS if col not in frame.columns]
    if missing:
        raise RuntimeError(f"Missing target columns in metadata: {missing}")
    return frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)


def append_query_chunk_to_graph(data: HeteroData, query_embeddings: np.ndarray, query_targets: np.ndarray, neighbor_idx: np.ndarray, neighbor_sims: np.ndarray) -> Tuple[HeteroData, int]:
    query_data = copy.deepcopy(data).cpu()
    train_count = int(query_data["image"].x.size(0))
    query_data["image"].x = torch.cat([query_data["image"].x.float(), torch.from_numpy(query_embeddings).float()], dim=0)
    if hasattr(query_data["image"], "y"):
        query_data["image"].y = torch.cat([query_data["image"].y.float(), torch.from_numpy(query_targets).float()], dim=0)
    else:
        query_data["image"].y = torch.from_numpy(query_targets).float()

    src, dst, weights = [], [], []
    for qi in range(neighbor_idx.shape[0]):
        query_node = train_count + qi
        for rank in range(neighbor_idx.shape[1]):
            src.append(query_node)
            dst.append(int(neighbor_idx[qi, rank]))
            weights.append(float(neighbor_sims[qi, rank]))
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    append_edge_index(query_data, ("image", "similar_to", "image"), edge_index, edge_weight)
    append_edge_index(query_data, ("image", "rev_similar_to", "image"), edge_index.flip(0), edge_weight)
    return query_data, train_count


def predict_test_queries(model: nn.Module, data_cpu: HeteroData, test_df: pd.DataFrame, test_embeddings: np.ndarray, top_k: int, label_indices: Sequence[int], chunk_size: int = QUERY_INFERENCE_CHUNK_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    train_embeddings = data_cpu["image"].x.cpu().numpy().astype("float32")
    test_targets = target_matrix_from_metadata(test_df)
    sims, idx = topk_inner_product(train_embeddings, test_embeddings, top_k=top_k)
    probs_chunks = []
    model.eval()
    for start in tqdm(range(0, len(test_embeddings), chunk_size), desc=f"test query chunks top_k={top_k}"):
        end = min(start + chunk_size, len(test_embeddings))
        query_data, train_count = append_query_chunk_to_graph(
            data_cpu,
            test_embeddings[start:end],
            test_targets[start:end],
            idx[start:end],
            sims[start:end],
        )
        query_edges = message_passing_edge_index_dict(query_data, include_has_finding=INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING)
        query_data = query_data.to(DEVICE)
        query_edges = {etype: edge_index.to(DEVICE) for etype, edge_index in query_edges.items()}
        with torch.no_grad():
            z = model(query_data.x_dict, query_edges)
            query_logits_all = model.decode_image_label_logits(z, label_indices)
            logits = query_logits_all[train_count:train_count + (end - start)].detach().cpu().numpy().astype("float32")
            probs_chunks.append(sigmoid_np(logits))
        del query_data, query_edges, z, query_logits_all
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.concatenate(probs_chunks, axis=0), test_targets

## Train and Evaluate One Fold

In [7]:
def experiment_root(model_name: str, query_top_k: int) -> Path:
    root = OUTPUT_ROOT / model_name / f"top_k_{query_top_k}"
    root.mkdir(parents=True, exist_ok=True)
    return root


def fold_dirs(model_name: str, query_top_k: int, fold: int) -> Dict[str, Path]:
    root = experiment_root(model_name, query_top_k) / f"fold{fold}"
    dirs = {
        "root": root,
        "models": root / "models",
        "metrics": root / "metrics",
        "predictions": root / "predictions",
        "logs": root / "logs",
    }
    for path in dirs.values():
        path.mkdir(parents=True, exist_ok=True)
    return dirs


def fold_complete(model_name: str, query_top_k: int, fold: int, dirs: Dict[str, Path]) -> bool:
    metrics_file = dirs["metrics"] / f"{model_name}_test_metrics.csv"
    tuned_file = dirs["metrics"] / f"{model_name}_test_metrics_val_tuned_thresholds.csv"
    return (dirs["root"] / "FOLD_COMPLETE.json").exists() and metrics_file.exists() and tuned_file.exists()


def save_predictions(path: Path, test_df: pd.DataFrame, probs: np.ndarray, targets: np.ndarray) -> pd.DataFrame:
    rows = []
    for i, row in test_df.reset_index(drop=True).iterrows():
        out = {
            "study_id": str(row.get("study_id", "")),
            "subject_id": str(row.get("subject_id", "")),
            "image_path": str(row.get("image_path", "")),
        }
        for j, label in enumerate(SCORING_LABELS):
            out[f"prob_{label}"] = float(probs[i, j])
            out[f"target_{label}"] = int(targets[i, j] >= 0.5)
        rows.append(out)
    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(path, index=False)
    return pred_df


def train_one_fold(model_name: str, query_top_k: int, fold: int) -> List[Dict]:
    model_name = model_name.lower()
    method_name = f"hetero_{model_name}"
    dirs = fold_dirs(model_name, query_top_k, fold)
    force_key = (model_name, int(query_top_k), int(fold))
    if SKIP_COMPLETED and force_key not in FORCE_RERUN and fold_complete(model_name, query_top_k, fold, dirs):
        print(f"{model_name} top_k={query_top_k} fold={fold}: already complete, skipping.")
        summary_csv = dirs["metrics"] / f"{model_name}_method_summary.csv"
        if summary_csv.exists():
            return pd.read_csv(summary_csv).to_dict(orient="records")
        summary_path = dirs["metrics"] / f"{model_name}_summary.json"
        return list(json.loads(summary_path.read_text()).values()) if summary_path.exists() else [{"fold": fold, "skipped": True}]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    cfg = get_model_config(model_name)
    print(f"Config for {model_name}:", cfg)

    data, train_df, test_df, test_embeddings = load_fold_data(fold)
    data = cap_similar_edges_in_data(data, cfg.get("max_train_similar_edges_per_node"))
    label_indices = label_indices_for_scoring(data)
    if not hasattr(data["image"], "y"):
        raise RuntimeError("data['image'].y is missing. Regenerate fold KG with train labels.")

    num_images = int(data["image"].x.size(0))
    train_idx, val_idx = split_train_val_indices(num_images, VAL_FRAC, seed=SEED + fold)
    y = data["image"].y.float()
    train_y = y[train_idx]
    val_y = y[val_idx]
    pos_weight = pos_weight_from_targets(train_y).to(DEVICE)

    mp_edges = message_passing_edge_index_dict(data, include_has_finding=INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING)
    metadata = (data.node_types, list(mp_edges.keys()))
    input_dims = input_dims_from_data(data)

    data = data.to(DEVICE)
    mp_edges = {etype: edge_index.to(DEVICE) for etype, edge_index in mp_edges.items()}

    model = build_model(model_name, metadata=metadata, input_dims=input_dims, cfg=cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_score = -float("inf")
    best_state = None
    best_epoch = -1
    stale_epochs = 0
    logs = []

    train_idx_dev = train_idx.to(DEVICE)
    val_idx_dev = val_idx.to(DEVICE)

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        z = model(data.x_dict, mp_edges)
        logits = model.decode_image_label_logits(z, label_indices)
        loss = criterion(logits[train_idx_dev], data["image"].y[train_idx_dev].float())
        loss.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            z = model(data.x_dict, mp_edges)
            logits = model.decode_image_label_logits(z, label_indices)
            val_logits = logits[val_idx_dev].detach().cpu().numpy().astype("float32")
            val_probs = sigmoid_np(val_logits)
            val_metrics = evaluate_prob_matrix(val_probs, val_y.numpy())
            val_macro_auroc = macro_metric(val_metrics, "auroc", disease_only=True)
            train_logits = logits[train_idx_dev].detach().cpu().numpy().astype("float32")
            train_probs = sigmoid_np(train_logits)
            train_metrics = evaluate_prob_matrix(train_probs, train_y.numpy())
            train_macro_auroc = macro_metric(train_metrics, "auroc", disease_only=True)

        log_row = {
            "fold": int(fold),
            "model_name": model_name,
            "query_top_k": int(query_top_k),
            "epoch": int(epoch),
            "loss": float(loss.detach().cpu()),
            "train_disease_macro_auroc": float(train_macro_auroc),
            "val_disease_macro_auroc": float(val_macro_auroc),
        }
        logs.append(log_row)
        print(f"model={model_name} top_k={query_top_k} fold={fold} epoch={epoch} loss={log_row['loss']:.4f} val_auc={val_macro_auroc:.4f}")

        if np.isfinite(val_macro_auroc) and val_macro_auroc > best_score:
            best_score = float(val_macro_auroc)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = int(epoch)
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"{model_name} top_k={query_top_k} fold={fold}: early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    pd.DataFrame(logs).to_csv(dirs["logs"] / f"{model_name}_training_log.csv", index=False)

    # Tune thresholds on validation nodes.
    model.eval()
    with torch.no_grad():
        z = model(data.x_dict, mp_edges)
        all_logits = model.decode_image_label_logits(z, label_indices)
        val_probs = sigmoid_np(all_logits[val_idx_dev].detach().cpu().numpy().astype("float32"))
    thresholds = tune_f1_thresholds(val_probs, val_y.numpy())
    pd.DataFrame({"label": SCORING_LABELS, "threshold": thresholds}).to_csv(dirs["metrics"] / f"{model_name}_val_f1_thresholds.csv", index=False)

    # Test-time query graph. Reload CPU graph to avoid carrying CUDA graph state.
    data_cpu, _, test_df, test_embeddings = load_fold_data(fold)
    data_cpu = cap_similar_edges_in_data(data_cpu, cfg.get("max_train_similar_edges_per_node"))
    test_probs, test_targets = predict_test_queries(
        model,
        data_cpu,
        test_df,
        test_embeddings,
        top_k=query_top_k,
        label_indices=label_indices,
        chunk_size=int(cfg.get("query_inference_chunk_size", QUERY_INFERENCE_CHUNK_SIZE)),
    )

    metrics_fixed = evaluate_prob_matrix(test_probs, test_targets)
    metrics_tuned = evaluate_prob_matrix(test_probs, test_targets, thresholds=thresholds)
    fixed_metrics_path = dirs["metrics"] / f"{model_name}_test_metrics.csv"
    tuned_metrics_path = dirs["metrics"] / f"{model_name}_test_metrics_val_tuned_thresholds.csv"
    metrics_fixed.to_csv(fixed_metrics_path, index=False)
    metrics_tuned.to_csv(tuned_metrics_path, index=False)
    save_predictions(dirs["predictions"] / f"{model_name}_test_predictions.csv", test_df, test_probs, test_targets)

    model_path = dirs["models"] / f"hetero_{model_name}_fold{fold}_topk{query_top_k}.pt"
    torch.save({
        "state_dict": model.state_dict(),
        "fold": int(fold),
        "model_name": model_name,
        "labels": list(SCORING_LABELS),
        "label_indices": list(label_indices),
        "input_dims": input_dims,
        "hidden_dim": int(cfg["hidden_dim"]),
        "num_layers": int(cfg["num_layers"]),
        "num_heads": int(cfg["num_heads"]),
        "max_train_similar_edges_per_node": cfg.get("max_train_similar_edges_per_node"),
        "dropout": DROPOUT,
        "decoder": DECODER,
        "include_has_finding_in_message_passing": INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING,
        "query_top_k": int(query_top_k),
        "best_epoch": int(best_epoch),
        "best_val_disease_macro_auroc": float(best_score),
    }, model_path)

    common_extra = {
        "model_name": model_name,
        "query_top_k": int(query_top_k),
        "include_has_finding_in_message_passing": bool(INCLUDE_HAS_FINDING_IN_MESSAGE_PASSING),
        "model_path": str(model_path),
        "best_epoch": int(best_epoch),
        "best_val_disease_macro_auroc": float(best_score),
        "hidden_dim": int(cfg["hidden_dim"]),
        "num_layers": int(cfg["num_layers"]),
        "num_heads": int(cfg["num_heads"]),
        "max_train_similar_edges_per_node": cfg.get("max_train_similar_edges_per_node"),
    }
    summary = metric_summary(metrics_fixed, fold, method_name, extra={
        **common_extra,
        "threshold_mode": "fixed_0.5",
        "metrics_path": str(fixed_metrics_path),
    })
    summary_tuned = metric_summary(metrics_tuned, fold, method_name, extra={
        **common_extra,
        "threshold_mode": "val_tuned_f1",
        "metrics_path": str(tuned_metrics_path),
    })
    pd.DataFrame([summary, summary_tuned]).to_csv(dirs["metrics"] / f"{model_name}_method_summary.csv", index=False)
    save_json({"fixed_0.5": summary, "val_tuned_f1": summary_tuned}, dirs["metrics"] / f"{model_name}_summary.json")
    save_json({
        "fold": int(fold),
        "status": "complete",
        "model_name": model_name,
        "query_top_k": int(query_top_k),
        "model_path": str(model_path),
    }, dirs["root"] / "FOLD_COMPLETE.json")
    return [summary, summary_tuned]

## Run 10-Fold Ablations

This cell runs `3 models × 4 top_k values × 10 folds`. It can take a long time. Because `SKIP_COMPLETED=True`, it is safe to interrupt and resume.

In [8]:
all_summaries = []

for model_name in MODEL_NAMES:
    for query_top_k in QUERY_TOP_K_VALUES:
        print(f"\n######## Model={model_name} top_k={query_top_k} ########")
        run_rows = []
        run_root = experiment_root(model_name, query_top_k)
        for fold in FOLD_INDICES:
            print(f"\n===== Model={model_name} top_k={query_top_k} fold={fold} =====")
            rows = train_one_fold(model_name, query_top_k, fold)
            run_rows.extend(rows)
            all_summaries.extend(rows)
            pd.DataFrame(run_rows).to_csv(run_root / f"{model_name}_topk{query_top_k}_10fold_summary_progress.csv", index=False)
            pd.DataFrame(all_summaries).to_csv(OUTPUT_ROOT / "transformer_gnn_10fold_all_results_progress.csv", index=False)
        pd.DataFrame(run_rows).to_csv(run_root / f"{model_name}_topk{query_top_k}_10fold_summary.csv", index=False)

all_results_df = pd.DataFrame(all_summaries)
all_results_df.to_csv(OUTPUT_ROOT / "transformer_gnn_10fold_all_results.csv", index=False)
display(all_results_df)


######## Model=hgtconv top_k=50 ########

===== Model=hgtconv top_k=50 fold=0 =====
hgtconv top_k=50 fold=0: already complete, skipping.

===== Model=hgtconv top_k=50 fold=1 =====
hgtconv top_k=50 fold=1: already complete, skipping.

===== Model=hgtconv top_k=50 fold=2 =====
hgtconv top_k=50 fold=2: already complete, skipping.

===== Model=hgtconv top_k=50 fold=3 =====
hgtconv top_k=50 fold=3: already complete, skipping.

===== Model=hgtconv top_k=50 fold=4 =====
hgtconv top_k=50 fold=4: already complete, skipping.

===== Model=hgtconv top_k=50 fold=5 =====
hgtconv top_k=50 fold=5: already complete, skipping.

===== Model=hgtconv top_k=50 fold=6 =====
hgtconv top_k=50 fold=6: already complete, skipping.

===== Model=hgtconv top_k=50 fold=7 =====
hgtconv top_k=50 fold=7: already complete, skipping.

===== Model=hgtconv top_k=50 fold=8 =====
hgtconv top_k=50 fold=8: already complete, skipping.

===== Model=hgtconv top_k=50 fold=9 =====
hgtconv top_k=50 fold=9: already complete, skipping

query FAISS top_k=50:   0%|          | 0/9 [00:00<?, ?it/s]

test query chunks top_k=50:   0%|          | 0/35 [00:00<?, ?it/s]


===== Model=transformerconv top_k=50 fold=1 =====
Config for transformerconv: {'hidden_dim': 128, 'num_layers': 1, 'num_heads': 2, 'dropout': 0.2, 'max_train_similar_edges_per_node': 50, 'query_inference_chunk_size': 128}
Capped ('image', 'similar_to', 'image'): 793800 -> 793800 edges
Capped ('image', 'rev_similar_to', 'image'): 793800 -> 719209 edges
model=transformerconv top_k=50 fold=1 epoch=0 loss=5.7026 val_auc=0.5000
model=transformerconv top_k=50 fold=1 epoch=1 loss=38.4121 val_auc=0.5002
model=transformerconv top_k=50 fold=1 epoch=2 loss=30.6416 val_auc=0.7527
model=transformerconv top_k=50 fold=1 epoch=3 loss=10.9973 val_auc=0.7664
model=transformerconv top_k=50 fold=1 epoch=4 loss=6.0571 val_auc=0.7673
model=transformerconv top_k=50 fold=1 epoch=5 loss=7.5592 val_auc=0.7688
model=transformerconv top_k=50 fold=1 epoch=6 loss=6.5128 val_auc=0.7705
model=transformerconv top_k=50 fold=1 epoch=7 loss=4.2421 val_auc=0.7711
model=transformerconv top_k=50 fold=1 epoch=8 loss=5.7969 

query FAISS top_k=50:   0%|          | 0/9 [00:00<?, ?it/s]

...

test query chunks top_k=500:   0%|          | 0/35 [00:00<?, ?it/s]


===== Model=transformerconv top_k=500 fold=9 =====
Config for transformerconv: {'hidden_dim': 128, 'num_layers': 1, 'num_heads': 2, 'dropout': 0.2, 'max_train_similar_edges_per_node': 50, 'query_inference_chunk_size': 128}
Capped ('image', 'similar_to', 'image'): 793800 -> 793800 edges
Capped ('image', 'rev_similar_to', 'image'): 793800 -> 718994 edges
model=transformerconv top_k=500 fold=9 epoch=0 loss=6.5812 val_auc=0.6927
model=transformerconv top_k=500 fold=9 epoch=1 loss=28.2920 val_auc=0.7219
model=transformerconv top_k=500 fold=9 epoch=2 loss=21.9694 val_auc=0.7177
model=transformerconv top_k=500 fold=9 epoch=3 loss=8.4224 val_auc=0.6980
model=transformerconv top_k=500 fold=9 epoch=4 loss=6.6233 val_auc=0.7159
model=transformerconv top_k=500 fold=9 epoch=5 loss=7.1836 val_auc=0.7382
model=transformerconv top_k=500 fold=9 epoch=6 loss=5.2840 val_auc=0.7511
model=transformerconv top_k=500 fold=9 epoch=7 loss=4.7431 val_auc=0.7589
model=transformerconv top_k=500 fold=9 epoch=8 los

query FAISS top_k=500:   0%|          | 0/9 [00:00<?, ?it/s]

test query chunks top_k=500:   0%|          | 0/35 [00:00<?, ?it/s]

,fold,method,disease_macro_auroc,disease_macro_auprc,disease_macro_f1,all_macro_auroc,all_macro_auprc,all_macro_f1,num_scored_labels,model_name,...,include_has_finding_in_message_passing,model_path,best_epoch,best_val_disease_macro_auroc,hidden_dim,num_layers,num_heads,threshold_mode,metrics_path,max_train_similar_edges_per_node
0,0,hetero_hgtconv,0.721302,0.114645,0.116580,0.700900,0.148488,0.164838,14,hgtconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,80,0.824205,256,2,4,fixed_0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,NaN
1,0,hetero_hgtconv,0.721302,0.114645,0.035305,0.700900,0.148488,0.089415,14,hgtconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,80,0.824205,256,2,4,val_tuned_f1,/data/liangz2/openi/multi_kg/kg_transformer_gn...,NaN
2,1,hetero_hgtconv,0.771815,0.147615,0.065293,0.733847,0.175119,0.117114,14,hgtconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,37,0.814668,256,2,4,fixed_0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,NaN
3,1,hetero_hgtconv,0.771815,0.147615,0.011913,0.733847,0.175119,0.067547,14,hgtconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,37,0.814668,256,2,4,val_tuned_f1,/data/liangz2/openi/multi_kg/kg_transformer_gn...,NaN
4,2,hetero_hgtconv,0.753678,0.134204,0.092787,0.737006,0.171997,0.142644,14,hgtconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,74,0.838662,256,2,4,fixed_0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,7,hetero_transformerconv,0.773265,0.138618,0.088419,0.773863,0.188668,0.140739,14,transformerconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,98,0.837674,128,1,2,val_tuned_f1,/data/liangz2/openi/multi_kg/kg_transformer_gn...,50.0
236,8,hetero_transformerconv,0.767554,0.143248,0.165308,0.770348,0.195102,0.212266,14,transformerconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,99,0.824925,128,1,2,fixed_0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,50.0
237,8,hetero_transformerconv,0.767554,0.143248,0.051484,0.770348,0.195102,0.107356,14,transformerconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,99,0.824925,128,1,2,val_tuned_f1,/data/liangz2/openi/multi_kg/kg_transformer_gn...,50.0
238,9,hetero_transformerconv,0.771768,0.142709,0.041440,0.773855,0.195686,0.091708,14,transformerconv,...,False,/data/liangz2/openi/multi_kg/kg_transformer_gn...,99,0.828717,128,1,2,fixed_0.5,/data/liangz2/openi/multi_kg/kg_transformer_gn...,50.0


## Aggregate Results for Tables

In [9]:
def collect_transformer_gnn_results(output_root: Path) -> pd.DataFrame:
    """Collect completed transformer-GNN rows from final, progress, run-level, or fold-level files."""
    candidate_files = []

    final_path = output_root / "transformer_gnn_10fold_all_results.csv"
    progress_path = output_root / "transformer_gnn_10fold_all_results_progress.csv"
    if final_path.exists():
        candidate_files.append(final_path)
    if progress_path.exists():
        candidate_files.append(progress_path)

    candidate_files.extend(sorted(output_root.glob("*/top_k_*/*_10fold_summary.csv")))
    candidate_files.extend(sorted(output_root.glob("*/top_k_*/*_10fold_summary_progress.csv")))
    candidate_files.extend(sorted(output_root.glob("*/top_k_*/fold*/metrics/*_method_summary.csv")))

    frames = []
    seen = set()
    for path in candidate_files:
        if path in seen or not path.exists():
            continue
        seen.add(path)
        try:
            frame = pd.read_csv(path)
        except Exception as exc:
            print(f"Skipping unreadable file {path}: {exc}")
            continue
        if frame.empty or "fold" not in frame.columns or "threshold_mode" not in frame.columns:
            continue
        if "model_name" not in frame.columns:
            # Infer from path like OUTPUT_ROOT / model_name / top_k_100 / ...
            try:
                frame["model_name"] = path.relative_to(output_root).parts[0]
            except Exception:
                frame["model_name"] = frame.get("method", "unknown")
        if "query_top_k" not in frame.columns:
            top_k = None
            for part in path.parts:
                if part.startswith("top_k_"):
                    top_k = int(part.split("top_k_", 1)[1])
                    break
            frame["query_top_k"] = top_k
        frame["source_file"] = str(path)
        frames.append(frame)

    if not frames:
        raise FileNotFoundError(
            "No transformer-GNN result files found. Run at least one fold in the ablation cell first. "
            f"Searched under: {output_root}"
        )

    results = pd.concat(frames, ignore_index=True)
    # De-duplicate repeated rows collected from both progress and final files.
    key_cols = [c for c in ["model_name", "query_top_k", "fold", "threshold_mode"] if c in results.columns]
    if key_cols:
        results = results.sort_values("source_file").drop_duplicates(subset=key_cols, keep="last")
    return results.reset_index(drop=True)


results = collect_transformer_gnn_results(OUTPUT_ROOT)
all_results_path = OUTPUT_ROOT / "transformer_gnn_10fold_all_results_collected.csv"
results.to_csv(all_results_path, index=False)
print("Collected rows:", len(results))
print("Saved collected long table:", all_results_path)

metric_cols = [
    "disease_macro_auroc", "disease_macro_auprc", "disease_macro_f1",
    "all_macro_auroc", "all_macro_auprc", "all_macro_f1",
]
missing_metrics = [col for col in metric_cols if col not in results.columns]
if missing_metrics:
    raise RuntimeError(f"Missing metric columns in collected results: {missing_metrics}")

summary_rows = []
for (model_name, query_top_k, threshold_mode), group in results.groupby(["model_name", "query_top_k", "threshold_mode"], sort=True):
    row = {
        "model_name": model_name,
        "method": group["method"].iloc[0] if "method" in group.columns else f"hetero_{model_name}",
        "query_top_k": int(query_top_k),
        "threshold_mode": threshold_mode,
        "n_folds": int(group["fold"].nunique()),
        "completed_folds": ",".join(str(int(x)) for x in sorted(group["fold"].dropna().unique())),
        "include_has_finding_in_message_passing": bool(group["include_has_finding_in_message_passing"].iloc[0]) if "include_has_finding_in_message_passing" in group.columns else False,
    }
    for metric in metric_cols:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=1)
    summary_rows.append(row)

aggregate = pd.DataFrame(summary_rows)
aggregate_path = OUTPUT_ROOT / "transformer_gnn_10fold_aggregate_summary.csv"
aggregate.to_csv(aggregate_path, index=False)
print("Saved aggregate summary:", aggregate_path)
display(aggregate.sort_values(["threshold_mode", "disease_macro_auroc_mean"], ascending=[True, False]))

Collected rows: 240
Saved collected long table: /data/liangz2/openi/multi_kg/kg_transformer_gnn_10fold/transformer_gnn_10fold_all_results_collected.csv
Saved aggregate summary: /data/liangz2/openi/multi_kg/kg_transformer_gnn_10fold/transformer_gnn_10fold_aggregate_summary.csv


,model_name,method,query_top_k,threshold_mode,n_folds,completed_folds,include_has_finding_in_message_passing,disease_macro_auroc_mean,disease_macro_auroc_std,disease_macro_auprc_mean,disease_macro_auprc_std,disease_macro_f1_mean,disease_macro_f1_std,all_macro_auroc_mean,all_macro_auroc_std,all_macro_auprc_mean,all_macro_auprc_std,all_macro_f1_mean,all_macro_f1_std
18,transformerconv,hetero_transformerconv,100,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.770436,0.006498,0.144542,0.004812,0.164655,0.032626,0.761113,0.016541,0.188383,0.009998,0.198711,0.040346
22,transformerconv,hetero_transformerconv,500,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.768704,0.007680,0.144791,0.004352,0.148816,0.049969,0.760038,0.024783,0.188929,0.013171,0.183767,0.060397
16,transformerconv,hetero_transformerconv,50,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.767784,0.007786,0.144768,0.004575,0.119084,0.075968,0.760067,0.016531,0.187605,0.008122,0.157347,0.086729
20,transformerconv,hetero_transformerconv,200,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.766991,0.009821,0.143559,0.006734,0.125617,0.068586,0.757595,0.019068,0.186682,0.011385,0.156708,0.080743
12,hgtconv,hetero_hgtconv,200,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.753658,0.017604,0.138588,0.010191,0.041748,0.056990,0.738013,0.028826,0.177262,0.016989,0.086959,0.058923
10,hgtconv,hetero_hgtconv,100,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.752076,0.016242,0.130203,0.011412,0.053095,0.054758,0.739599,0.020814,0.170634,0.013394,0.087663,0.061102
8,hgtconv,hetero_hgtconv,50,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.749005,0.016486,0.135980,0.011523,0.050116,0.050804,0.730742,0.016132,0.172279,0.013718,0.092385,0.060804
4,hanconv,hetero_hanconv,200,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.740459,0.016762,0.145870,0.005947,0.058952,0.070584,0.746079,0.015654,0.198245,0.005693,0.111229,0.065542
0,hanconv,hetero_hanconv,50,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.729423,0.050156,0.141749,0.021500,0.066345,0.057473,0.732616,0.057102,0.192146,0.027475,0.118095,0.053372
6,hanconv,hetero_hanconv,500,fixed_0.5,10,"0,1,2,3,4,5,6,7,8,9",False,0.716492,0.078828,0.136100,0.030896,0.014210,0.031644,0.721983,0.078035,0.187963,0.032066,0.064061,0.036710


## Notes

Use `fixed_0.5` as the primary threshold-free comparison companion to AUROC/AUPRC, and use `val_tuned_f1` only to report threshold-calibrated F1. The ranking metrics AUROC and AUPRC do not change when thresholds are tuned.

For paper tables, compare this notebook against:

- BiomedCLIP + MLP.
- FAISS retrieval only.
- BiomedCLIP + FAISS fusion.
- BiomedCLIP + FAISS + RoentGen diffusion augmentation.
- Hetero GraphSAGE.
- Transformer-GNN variants in this notebook.